In [20]:
from dotenv import load_dotenv
import os
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set")

print("API key loaded")

API key loaded


In [13]:
import git
import os

repo_url = "https://github.com/abachaa/MedQuAD.git"
repo_dir = "MedQuAD"

if not os.path.exists(repo_dir):
    git.Repo.clone_from(repo_url, repo_dir)

print("Dataset downloaded")

Dataset downloaded


In [14]:
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

documents = []

root_dir = "MedQuAD"

for folder in os.listdir(root_dir):

    folder_path = os.path.join(root_dir, folder)

    if not os.path.isdir(folder_path):
        continue

    xml_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".xml")
    ]

    for xml_file in tqdm(xml_files):

        xml_path = os.path.join(folder_path, xml_file)

        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            for qa in root.findall(".//QAPair"):

                question = qa.findtext("Question")
                answer = qa.findtext("Answer")

                if question and answer:

                    documents.append(
                        {
                            "question": question.strip(),
                            "answer": answer.strip(),
                            "source": folder
                        }
                    )

        except Exception as e:
            print(f"Error parsing {xml_file}: {e}")

print(f"Loaded {len(documents)} QA pairs")

100%|██████████| 59/59 [00:00<00:00, 19707.25it/s]
0it [00:00, ?it/s]
100%|██████████| 1312/1312 [00:00<00:00, 28774.98it/s]

Loaded 16407 QA pairs


In [27]:
documents[2485]


{'question': 'What is (are) pyridoxine-dependent epilepsy ?',
 'answer': 'Pyridoxine-dependent epilepsy is a condition that involves seizures beginning in infancy or, in some cases, before birth. Those affected typically experience prolonged seizures lasting several minutes (status epilepticus). These seizures involve muscle rigidity, convulsions, and loss of consciousness (tonic-clonic seizures). Additional features of pyridoxine-dependent epilepsy include low body temperature (hypothermia), poor muscle tone (dystonia) soon after birth, and irritability before a seizure episode. In rare instances, children with this condition do not have seizures until they are 1 to 3 years old.  Anticonvulsant drugs, which are usually given to control seizures, are ineffective in people with pyridoxine-dependent epilepsy. Instead, people with this type of seizure are medically treated with large daily doses of pyridoxine (a type of vitamin B6 found in food). If left untreated, people with this condit

In [28]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

In [30]:
from langchain_core.documents import Document
langchain_documents = [
    Document(
        page_content=f"Question: {doc['question']}\nAnswer: {doc['answer']}",
        metadata={"source": doc["source"]}
    )
    for doc in documents
]
vector_db = Chroma.from_documents(
    documents=langchain_documents,
    embedding=embeddings,
    persist_directory="./medquad_chroma"
)

In [31]:
query = "What are symptoms of diabetes?"

results = vector_db.similarity_search(
    query,
    k=5
)

for idx, doc in enumerate(results):

    print("=" * 80)
    print(f"Result {idx+1}")
    print(doc.page_content[:1000])

Result 1
Question: What are the symptoms of Diabetes ?
Answer: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some lose weight without trying. Additional signs include sores that heal slowly, dry, itchy skin, loss of feeling or tingling in the feet and blurry eyesight. Some people with diabetes, however, have no symptoms at all.
Result 2
Question: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2 ?
Answer: The signs and symptoms of diabetes are
                
- being very thirsty  - urinating often  - feeling very hungry  - feeling very tired  - losing weight without trying  - sores that heal slowly  - dry, itchy skin  - feelings of pins and needles in your feet  - losing feeling in your feet  - blurry eyesight
                
Some people with diabetes dont have any of these signs or symptoms. The only way to know if you have diabetes is to have your doctor do a blood test.
Re

In [32]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [33]:
docs = retriever.invoke(
    "What causes hypertension?"
)

for d in docs:
    print(d.page_content)

Question: What causes High Blood Pressure ?
Answer: Changes, either fromgenesor the environment, in the bodys normal functions may cause high blood pressure, including changes to kidney fluid and salt balances, therenin-angiotensin-aldosterone system,sympathetic nervous systemactivity, and blood vessel structure and function.
                
Biology and High Blood Pressure
                
Researchers continue to study how various changes in normal body functions cause high blood pressure. The key functions affected in high blood pressure include:
                
Kidney fluid and salt balances
                
Renin-angiotensin-aldosterone system
                
Sympathetic nervous system activity
                
Blood vessel structure and function
                

                
Kidney Fluid and Salt Balances
                
The kidneys normally regulate the bodys salt balance by retaining sodium and water and excreting potassium. Imbalances in this kidney function can expand 